# Acceder a datos de calidad del aire casi en tiempo real (NRT) de TEMPO

Los archivos de dióxido de nitrógeno Nivel 2 (PROVISIONAL) proporcionan información de gases traza a la resolución espacial nativa de TEMPO, ~10 km^2 en el centro del Field of Regard (FOR), para gránulos individuales. Cada gránulo cubre todo el FOR Norte-Sur de TEMPO, pero solo una parte del FOR Este-Oeste. **Fuente:** [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/larc-cloud-tempo-no2-l2-v04).

## <span style='color:#ff6666'> **Requisitos**
1. Autenticación EDL (usuario/contraseña)
2. `pydap>=3.5.9`.

## <span style='color:#ff6666'>**Objetivos**

### Subdividir un archivo remoto

- **a)** Por variables
- **b)** Por selección espacial

### Subdividir múltiples archivos remotos

- Transmitir subconjunto de datos

### <span style='color:#ff6666'> **Referencias**

**Liu, X. (2025)**. <i>TEMPO NO2 tropospheric, stratospheric, and total columns V04</i> [Data set]. NASA Langley Atmospheric Science Data Center Distributed Active Archive Center. https://doi.org/10.5067/IS-40E/TEMPO/NO2_L2.004


In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import pydap
import matplotlib.pyplot as plt
# import pydap-specific tools
from pydap.net import create_session
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf
import numpy as np

## Encontrar URLs OPeNDAP

### **Consultar URLs opendap usando la API CMR de NASA**


Nos interesan datos de `TEMPO NO2 tropospheric and stratospheric columns V04`. Esta colección proporciona datos horarios para datos de nivel 2, considerados Near Real Time (NRT).


In [ ]:
TEMPO_L2_NRTNO2_ccid = "C3685896872-LARC_CLOUD" # 
time_range = [dt.datetime(2025, 10, 1), dt.datetime(2025, 10, 7)] # One month of data

bounding_box = [-124.63309,46.35932,  -121, 49.83307] # WSEN area within Seattle PNW

cmr_urls = get_cmr_urls(ccid=TEMPO_L2_NRTNO2_ccid, bounding_box=bounding_box, time_range=time_range, limit=1000) # you can incread the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[:5]

## Autenticación EDL mediante earthaccess y OPeNDAP

Puedes autenticarte mediante earthaccess como se muestra abajo. Debes tener una cuenta EDL válida. Hay dos estrategias para autenticarte con `earthaccess`:

1. `strategy="interactive"`. Esto pedirá tu usuario y contraseña de EDL.
2. `strategy="netrc"`. Usa esta opción si el notebook se ejecuta en un entorno donde se puede recuperar un `.netrc` con tus credenciales.

A continuación, el valor predeterminado será `netrc`, asumiendo que el usuario ejecutó el notebook **Authenticate.ipynb**. Si no es así, puedes cambiar la estrategia a `"interactive"`.


In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


## Acceso SOLO a metadatos con PyDAP

Podemos acceder a metadatos producidos por <span style='color:#ff6666'>**OPeNDAP**</span> para identificar las variables de interés. En particular, aquellas asociadas con valores de latitud y longitud.

A continuación necesitamos solicitar los metadatos <span style='color:#ff6666'>**DAP4**</span> al servidor remoto.


In [ ]:
%%time
pyds = open_url(cmr_urls[0],protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
dims = list(set(pyds['geolocation/latitude'].dims + pyds['geolocation/longitude'].dims + pyds['geolocation/time'].dims))

In [ ]:
print("\nnecessary dimensions to download:", dims, "\n")

### Subdividir por nombres de variables

<font size="3.5"> Primero exploramos solo las coordenadas y sus dimensiones, para identificar el subconjunto espacial.


In [ ]:
output_path = "data/"

## Transmitir datos

<font size="3.5"> Cada archivo remoto se almacena en un archivo individual. No hay agregación de datos.


In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, 
              keep_variables= dims + [
                                      "/geolocation/time",
                                      "/geolocation/longitude",
                                      "/geolocation/latitude", 
              ],
              output_path=output_path)

## Inspeccionar todos los archivos descargados

<font size="3"> Aquí identificamos con más detalle el subconjunto de datos necesario en el archivo remoto, que devolverá SOLO datos dentro de nuestra caja delimitadora, para cualquier variable de interés posible.


In [ ]:
%%time
# Use coord data from Bounding Box
minLon, maxLon = bounding_box[0], bounding_box[2]
minLat, maxLat = bounding_box[1], bounding_box[3]

slices = []
# iterate over all downloaded files
# Will use the URL to extract the filename
for url in cmr_urls:
    filename = output_path+f"{url.split('/')[-1][:-3]}.nc4"
    # Flatten data 
    ds = xr.merge([xr.open_dataset(filename), xr.open_dataset(filename, group='geolocation')])
    ds.load()
    # Identify subset from Lon/Lat data per granule
    
    longitude = ds['longitude'].values
    latitude = ds['latitude'].values

    mask = (
        (longitude >= minLon) & (longitude <= maxLon) &
        (latitude >= minLat) & (latitude <= maxLat)
    )

    rows, cols = np.where(mask)
    # indexes below
    y0, y1 = rows.min(), rows.max()
    x0, x1 = cols.min(), cols.max()
    slice_ = {
        "mirror_step":(y0,y1),
        "xtrack": (x0,x1),
        }
    slices.append({
        "mirror_step":(y0,y1),
        "xtrack": (x0,x1),
        })

### Visualizar coordenadas

<font size="3.5"> Será necesario enmascarar arreglos para visualizar,

<font size="3.5"> Graficar solo el último gránulo.


In [ ]:
Lon = ds['longitude'].isel(mirror_step=slice(y0, y1), xtrack=slice(x0, x1))

Lon_masked = xr.full_like(ds['longitude'], np.nan)
Lon_masked.loc[dict(
    mirror_step=Lon['mirror_step'],
    xtrack=Lon['xtrack']
)] = Lon


Lat = ds['latitude'].isel(mirror_step=slice(y0, y1), xtrack=slice(x0, x1))
Lat_masked = xr.full_like(ds['latitude'], np.nan)
Lat_masked.loc[dict(
    mirror_step=Lat['mirror_step'],
    xtrack=Lat['xtrack']
)] = Lat


In [ ]:
fig, axes = plt.subplots(figsize=(20,8), ncols=2)
pbar_lon = ds['longitude'].plot(ax=axes[0], cmap="Blues", vmin=-160, vmax=-105, levels=np.arange(-160,-105,3), cbar_kwargs={"location": "top"})
pbar_lon.colorbar.ax.tick_params(labelsize=14)
pbar_lon.colorbar.set_label(r'Longitude ($^\circ$E)', fontsize=16, weight='bold')
Lon_masked.plot(ax=axes[0], cmap="Greys_r",vmin=-160,vmax=20, add_colorbar=False, alpha=0.8)

# Optional: Set limits if not automatically handling it
axes[0].set_xlim([ds['xtrack'].min(),ds['xtrack'].max()])
axes[0].set_ylim([ds['mirror_step'].min(),ds['mirror_step'].max()])

pbar_lat = ds['latitude'].plot(ax=axes[1], vmin=15, vmax=62.5, levels=20, cmap='Reds',cbar_kwargs={"location": "top"})
pbar_lat.colorbar.ax.tick_params(labelsize=14)
pbar_lat.colorbar.set_label(r'Latitude ($^\circ$N)', fontsize=16, weight='bold')
Lat_masked.plot(ax=axes[1], cmap="Greys_r",vmin=40,vmax=90, add_colorbar=False, alpha=0.8)


plt.setp(axes[0].get_xticklabels(), fontsize=15)
plt.setp(axes[0].get_yticklabels(), fontsize=15)
axes[0].set_xlabel('xtrack', fontsize=17.5)
axes[0].set_ylabel('mirror_step', fontsize=17.5)

plt.setp(axes[1].get_xticklabels(), fontsize=15)
plt.setp(axes[1].get_yticklabels(), fontsize=15);
axes[1].set_xlabel('xtrack', fontsize=17.5)
axes[1].set_ylabel('mirror_step', fontsize=17.5)
plt.show()

## Ahora definir todas las variables para descargar


In [ ]:
Vars = dims + [
    "/product/main_data_quality_flag",
    "/product/vertical_column_troposphere",
    "/product/vertical_column_stratosphere",
    "/geolocation/time",
    "/geolocation/longitude",
    "/geolocation/latitude",
    "/support_data/wind_speed",
    "/support_data/terrain_height",
    "/support_data/gas_profile",
    "/support_data/pbl_height",
    "/support_data/temperature_profile",
]

## Descargar datos

<font size="3.5"> En este momento, es necesario borrar cualquier dato `TEMPO_NO2_L2_*` descargado previamente

<font size="3.5"> para evitar colisión de nombres de archivo.


In [ ]:
import os
import glob

fnames = [output_path+f"{fname.split('/')[-1][:-3]}.nc4" for fname in cmr_urls]

for filename in fnames:
    try:
        os.remove(filename)
    except FileNotFoundError:
        print(f"The file '{filename}' is not in there anymore")    

In [ ]:
%%time
dap_to_netcdf(cmr_urls, session=my_session, 
              keep_variables = Vars,
              dim_slices= slices,
              output_path=output_path)

In [ ]:
local_file = output_path+cmr_urls[0].split("/")[-1][:-3]+".nc4"
dst = xr.open_datatree(local_file)
dst